# Test Whisper Base vs Whisper + LoRA

**Purpose:** Compare base Whisper-large-v3 against LoRA-enhanced version

**Question:** Are the LoRA adapters (trained on synthetic TTS) actually helpful?

---

## Setup Requirements

### Kaggle Settings:
- ✅ **GPU**: T4 x2 (free tier is fine)
- ✅ **Internet**: ON (to download models)
- ✅ **Persistence**: OFF (we don't need it)

### Data Requirements:
1. **Audio files**: Upload 2-3 real Arabic medical audio files to test
2. **LoRA adapters**: Upload your `lora_ckpt` folder as a Kaggle dataset

### Expected Runtime:
- Cell 1 (install): ~3-5 minutes
- Cell 2 (load models): ~2-3 minutes  
- Cell 3 (test): ~1-2 minutes per audio file
- **Total: ~10-15 minutes**

## Cell 1: Install Dependencies

**Time:** ~3-5 minutes

**What it does:**
- Installs transformers, peft, librosa
- Compatible versions for Kaggle (CUDA 12.2)
- Includes jiwer for WER calculation

In [ ]:
%%time
import subprocess
import sys

print("📦 Installing dependencies for Kaggle...")
print()

# Kaggle-compatible versions
packages = [
    'transformers==4.36.2',      # Stable, works with CUDA 12.2
    'peft==0.7.1',               # For LoRA loading
    'accelerate==0.25.0',        # For model loading
    'librosa==0.10.1',           # Audio loading
    'soundfile==0.12.1',         # Audio I/O
    'jiwer==3.0.3',              # WER calculation
]

for pkg in packages:
    print(f"Installing {pkg}...")
    subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '-q', pkg],
        check=False
    )

print()
print("✅ All packages installed!")
print()
print("Package versions:")
import transformers, peft, accelerate, librosa, jiwer
print(f"  transformers: {transformers.__version__}")
print(f"  peft: {peft.__version__}")
print(f"  accelerate: {accelerate.__version__}")
print(f"  librosa: {librosa.__version__}")
print(f"  jiwer: {jiwer.__version__}")
print()
print("⚠️  Do NOT restart kernel - proceed to next cell")

## Cell 2: Load Models

**Time:** ~2-3 minutes

**What it does:**
1. Loads base Whisper-large-v3 (~3GB download first time)
2. Tries to load your LoRA adapters
3. Shows which GPU is being used

In [ ]:
%%time
import torch
from transformers import WhisperForConditionalGeneration, WhisperProcessor, pipeline
from peft import PeftModel
import warnings
warnings.filterwarnings('ignore')

print("=" * 80)
print("LOADING MODELS")
print("=" * 80)
print()

# Check GPU
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print()

# ============================================================================
# LOAD BASE WHISPER (using pipeline for simplicity)
# ============================================================================
print("📥 Loading base Whisper-large-v3...")
base_pipe = pipeline(
    "automatic-speech-recognition",
    model="openai/whisper-large-v3",
    device=0 if device == "cuda" else -1,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
)
print("✅ Base Whisper loaded (pipeline)")
print()

# ============================================================================
# LOAD WHISPER + LORA (if adapters exist)
# ============================================================================
# ⚠️ IMPORTANT: Update this path to match YOUR Kaggle dataset!
# Options:
#   1. If you uploaded lora_ckpt as a dataset named "whisper-lora":
#      LORA_PATH = "/kaggle/input/whisper-lora/lora_ckpt"
#   2. If you uploaded to working directory:
#      LORA_PATH = "/kaggle/working/lora_ckpt"
#   3. If testing without LoRA:
#      LORA_PATH = None

LORA_PATH = "/kaggle/input/whisper-lora/lora_ckpt"  # ⚠️ CHANGE THIS!

lora_model = None
lora_processor = None

if LORA_PATH:
    import os
    if os.path.exists(LORA_PATH):
        print(f"📥 Loading LoRA adapters from: {LORA_PATH}")
        try:
            # Load base model (for LoRA)
            base_model = WhisperForConditionalGeneration.from_pretrained(
                "openai/whisper-large-v3",
                torch_dtype=torch.float16 if device == "cuda" else torch.float32,
                device_map=device,
            )
            
            # Load LoRA adapters
            lora_model = PeftModel.from_pretrained(
                base_model,
                LORA_PATH,
                torch_dtype=torch.float16 if device == "cuda" else torch.float32
            )
            lora_model.eval()
            
            # Load processor
            lora_processor = WhisperProcessor.from_pretrained("openai/whisper-large-v3")
            
            print("✅ LoRA model loaded successfully!")
            print()
        except Exception as e:
            print(f"❌ Failed to load LoRA: {e}")
            print("Will only test base Whisper")
            print()
    else:
        print(f"⚠️  LoRA path not found: {LORA_PATH}")
        print("Will only test base Whisper")
        print()
else:
    print("ℹ️  LORA_PATH not set - testing base Whisper only")
    print()

print("=" * 80)
print("MODELS READY!")
print("=" * 80)
print()
print(f"Base Whisper: {'✅ Loaded' if base_pipe else '❌ Failed'}")
print(f"LoRA Whisper: {'✅ Loaded' if lora_model else '❌ Not available'}")
print()

## Cell 3: Test on Audio Files

**Time:** ~1-2 minutes per file

**What it does:**
1. Lists available audio files
2. Transcribes with base Whisper
3. Transcribes with LoRA Whisper (if loaded)
4. Compares outputs side-by-side
5. Calculates WER if you provide ground truth

In [ ]:
import os
import librosa
import time
from jiwer import wer, cer

print("=" * 80)
print("AUDIO FILE TEST")
print("=" * 80)
print()

# ============================================================================
# CONFIGURE TEST FILES
# ============================================================================
# Option 1: Auto-detect audio files
audio_extensions = ['.mp3', '.m4a', '.wav', '.flac', '.ogg']
test_files = []

# Search in /kaggle/input (your uploaded datasets)
for root, dirs, files in os.walk('/kaggle/input'):
    for file in files:
        if any(file.lower().endswith(ext) for ext in audio_extensions):
            test_files.append(os.path.join(root, file))

# Search in /kaggle/working (manually uploaded)
for root, dirs, files in os.walk('/kaggle/working'):
    for file in files:
        if any(file.lower().endswith(ext) for ext in audio_extensions):
            test_files.append(os.path.join(root, file))

# Option 2: Manually specify (uncomment and edit if needed)
# test_files = [
#     '/kaggle/input/my-audio/test1.mp3',
#     '/kaggle/input/my-audio/test2.m4a',
# ]

print(f"Found {len(test_files)} audio file(s):")
for f in test_files:
    size_mb = os.path.getsize(f) / (1024 * 1024)
    print(f"  📁 {os.path.basename(f)} ({size_mb:.1f} MB)")
print()

if len(test_files) == 0:
    print("⚠️  No audio files found!")
    print()
    print("Please:")
    print("1. Upload audio files to Kaggle")
    print("2. Add them as input datasets")
    print("3. Or manually specify paths in test_files list above")
    print()
else:
    # ========================================================================
    # OPTIONAL: Ground truth transcriptions for WER calculation
    # ========================================================================
    # Format: {"filename.mp3": "النص الصحيح هنا"}
    ground_truth = {
        # "test1.mp3": "أنا عندي ألم في البطن منذ يومين",
        # "test2.m4a": "التهاب اللثة يحتاج علاج فوري",
    }
    
    # ========================================================================
    # RUN TESTS
    # ========================================================================
    results = []
    
    for audio_file in test_files:
        filename = os.path.basename(audio_file)
        
        print("=" * 80)
        print(f"Testing: {filename}")
        print("=" * 80)
        print()
        
        # Get audio duration
        audio_data, sr = librosa.load(audio_file, sr=16000, mono=True)
        duration = len(audio_data) / sr
        print(f"Duration: {duration:.1f}s")
        print()
        
        # ====================================================================
        # TEST 1: Base Whisper
        # ====================================================================
        print("🎤 Transcribing with BASE Whisper...")
        start = time.time()
        base_result = base_pipe(
            audio_file,
            generate_kwargs={"language": "arabic", "task": "transcribe"}
        )
        base_time = time.time() - start
        base_text = base_result['text'].strip()
        base_rtf = base_time / duration
        
        print(f"✅ Done in {base_time:.2f}s (RTF: {base_rtf:.3f}x)")
        print(f"Text: {base_text}")
        print()
        
        # ====================================================================
        # TEST 2: LoRA Whisper (if available)
        # ====================================================================
        lora_text = None
        lora_time = None
        lora_rtf = None
        
        if lora_model and lora_processor:
            print("🎤 Transcribing with LORA Whisper...")
            start = time.time()
            
            # Process audio
            inputs = lora_processor(
                audio_data,
                sampling_rate=16000,
                return_tensors="pt"
            )
            input_features = inputs.input_features.to(device)
            
            # Generate
            with torch.no_grad():
                predicted_ids = lora_model.generate(
                    input_features=input_features,
                    language="ar",
                    task="transcribe",
                    max_new_tokens=448,
                )
            
            lora_time = time.time() - start
            lora_text = lora_processor.batch_decode(
                predicted_ids,
                skip_special_tokens=True
            )[0].strip()
            lora_rtf = lora_time / duration
            
            print(f"✅ Done in {lora_time:.2f}s (RTF: {lora_rtf:.3f}x)")
            print(f"Text: {lora_text}")
            print()
        
        # ====================================================================
        # COMPARISON
        # ====================================================================
        print("📊 COMPARISON:")
        print("-" * 80)
        
        if lora_text:
            # Check if identical
            if base_text == lora_text:
                print("⚠️  IDENTICAL TRANSCRIPTIONS")
                print("   LoRA adapters made NO difference!")
                print()
                print("   This suggests:")
                print("   - LoRA not being applied correctly, OR")
                print("   - LoRA trained on synthetic data that doesn't help real audio")
            else:
                print("✅ DIFFERENT TRANSCRIPTIONS")
                print()
                
                # Word-level differences
                base_words = base_text.split()
                lora_words = lora_text.split()
                
                print(f"Word count: Base={len(base_words)}, LoRA={len(lora_words)}")
                print()
                
                # Show first few differences
                print("Word differences:")
                diff_count = 0
                for i, (b, l) in enumerate(zip(base_words, lora_words)):
                    if b != l:
                        print(f"  Position {i+1}: '{b}' → '{l}'")
                        diff_count += 1
                        if diff_count >= 5:  # Show max 5 differences
                            print("  ...")
                            break
            print()
        
        # ====================================================================
        # WER CALCULATION (if ground truth provided)
        # ====================================================================
        gt_text = ground_truth.get(filename)
        if gt_text:
            print("📏 ACCURACY (vs Ground Truth):")
            print(f"Ground Truth: {gt_text}")
            print()
            
            base_wer = wer(gt_text, base_text) * 100
            base_cer = cer(gt_text, base_text) * 100
            
            print(f"Base Whisper:")
            print(f"  WER: {base_wer:.2f}%")
            print(f"  CER: {base_cer:.2f}%")
            print()
            
            if lora_text:
                lora_wer = wer(gt_text, lora_text) * 100
                lora_cer = cer(gt_text, lora_text) * 100
                
                print(f"LoRA Whisper:")
                print(f"  WER: {lora_wer:.2f}%")
                print(f"  CER: {lora_cer:.2f}%")
                print()
                
                # Improvement
                wer_diff = base_wer - lora_wer
                if wer_diff > 0:
                    print(f"✅ LoRA is BETTER by {wer_diff:.2f}% WER")
                elif wer_diff < 0:
                    print(f"❌ LoRA is WORSE by {abs(wer_diff):.2f}% WER")
                else:
                    print(f"⚖️  No WER difference")
                print()
        
        # Store results
        results.append({
            'file': filename,
            'duration': duration,
            'base_text': base_text,
            'base_time': base_time,
            'lora_text': lora_text,
            'lora_time': lora_time,
        })
        
        print()
    
    # ========================================================================
    # FINAL SUMMARY
    # ========================================================================
    print("=" * 80)
    print("FINAL VERDICT")
    print("=" * 80)
    print()
    
    if lora_model:
        identical_count = sum(1 for r in results if r['base_text'] == r['lora_text'])
        total = len(results)
        
        print(f"Tested {total} audio file(s)")
        print(f"Identical transcriptions: {identical_count}/{total}")
        print()
        
        if identical_count == total:
            print("❌ RECOMMENDATION: DO NOT USE LORA")
            print()
            print("Reasons:")
            print("- LoRA produced IDENTICAL results to base model")
            print("- Likely trained on synthetic TTS data (not real speech)")
            print("- Adds complexity without benefit")
            print()
            print("✅ Use base Whisper-large-v3 instead!")
        elif identical_count > 0:
            print("⚠️  MIXED RESULTS")
            print()
            print("- Some files: LoRA makes no difference")
            print("- Some files: LoRA changes output")
            print()
            print("Next step: Manually review transcriptions above")
            print("Check if LoRA changes are improvements or degradations")
        else:
            print("✅ LoRA IS MAKING CHANGES")
            print()
            print("Next step: Manually review transcriptions above")
            print("Determine if LoRA changes are meaningful improvements")
            print()
            print("Questions to ask:")
            print("- Are medical terms more accurate?")
            print("- Is dialect handling better?")
            print("- Is overall quality higher?")
    else:
        print("ℹ️  LoRA adapters were not tested")
        print()
        print("Base Whisper-large-v3 results shown above.")
        print("This model is already excellent for Arabic medical audio!")
    
    print()
    print("=" * 80)

## Summary

### If transcriptions are IDENTICAL:
❌ **Don't use LoRA adapters**
- They're not helping at all
- Likely trained on synthetic TTS data
- Use base Whisper-large-v3 instead

### If transcriptions are DIFFERENT but WORSE:
❌ **Don't use LoRA adapters**
- They're hurting accuracy
- Training data was wrong domain (TTS vs real speech)
- Use base Whisper-large-v3 instead

### If transcriptions are DIFFERENT and BETTER:
✅ **Keep using LoRA adapters!**
- They're improving medical terminology
- Training data was appropriate

---

## My Prediction:
If Salma trained on **AI-generated TTS audio**, the LoRA adapters will be **useless**.

**Base Whisper-large-v3 is already excellent for Arabic medical audio!**